<a href="https://colab.research.google.com/github/zyf-hitsz/PytorchLearning/blob/main/%E7%A5%9E%E7%BB%8F%E7%BD%91%E7%BB%9C/%E9%9D%9E%E7%BA%BF%E6%80%A7%E6%BF%80%E6%B4%BB%E5%B1%82Non_linearActivations%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#非线性激活
---
###Question1:为什么神经网络要非线性？
我们先看一个两层的神经网络：
$$h=W_1x+b_1,$$$$y=W_2x+b_2.$$
代入整理得到：$$y=(W_1W_2)x+(W_2b_1+b_2)$$
以此类推，如果每一层都是线性的，那么无论多少层最后一定也是线性的。我们使用的卷积本质上也是一个线性变换，所以如果没有非线性层，那么数学表达上还是等价于一个线性计算。加入非线性激活函数后，神经网络就可以表达：
*   曲线
*   分段函数
*   非线性边缘
*   高阶特征组合
*   极其复杂的输入输出映射

神经网络的基本结构可以理解为：线性变换+非线性变换+线性变换+非线性变换……
###Question2:激活函数实际上在做什么？
以某个神经元为例，线性层先得到：
$$z=w^Tx+b$$
激活函数进行：
$$a=f(z)$$
可以理解成：
线性层负责“计算某种特征响应”，激活函数决定“这个响应应该以什么方式继续向后传播，即进行特征选择”。
###Question3:怎样评价一个激活函数？
| 特性 | 意义 |
|---|---|
| 非线性 | 能否增加网络表达能力  |
| 梯度大小 | 是否容易发生梯度消失 |
| 是否饱和 | 大输入时梯度是否趋近 0 |
| 负半轴 | 是否允许负值传播 |
| 是否零中心 | 输出是否同时具有正负值 |
| 平滑性 | 是否处处可导 |
| 单调性 | 输入增大时输出是否一定增大 |
| 稀疏性 | 是否会产生大量 0 |
| 计算量 | sigmoid / exp / erf 通常比 ReLU 贵 |
| 是否有参数 | 如 PReLU 有可学习参数 |
| 数值稳定性 | 深层网络中是否稳定 |

现代激活函数的演进，大体就是在寻找：表达能力+梯度传播+计算成本之间更好的平衡。








In [50]:
import torch
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, Markdown, Math


x = torch.linspace(-6, 6, 1000)


def activation_card(
    name,
    activation,
    formula,
    description,
    xlim=(-6, 6),
    ylim=None
):
    # =========================
    # 左侧：文字 + 数学公式
    # =========================
    left_output = widgets.Output(
        layout=widgets.Layout(
            width="48%",
            padding="15px"
        )
    )

    with left_output:

        # 标题
        display(Markdown(f"## {name}"))

        # 小标题
        display(Markdown("**数学定义：**"))

        # 关键：直接用 IPython Math 渲染 LaTeX
        display(Math(formula))

        # 说明文字
        display(Markdown(description))


    # =========================
    # 右侧：函数图像
    # =========================
    right_output = widgets.Output(
        layout=widgets.Layout(
            width="52%"
        )
    )

    with right_output:

        with torch.no_grad():
            y = activation(x)

        fig, ax = plt.subplots(figsize=(5, 3.5))

        ax.plot(x.numpy(), y.numpy())

        ax.axhline(0, linewidth=0.8)
        ax.axvline(0, linewidth=0.8)

        ax.grid(alpha=0.25)

        ax.set_xlim(*xlim)

        if ylim is not None:
            ax.set_ylim(*ylim)

        ax.set_xlabel("x")
        ax.set_ylabel("f(x)")
        ax.set_title(name)

        plt.show()


    # =========================
    # 左右并排
    # =========================
    display(
        widgets.HBox(
            [left_output, right_output],
            layout=widgets.Layout(
                width="100%",
                align_items="center"
            )
        )
    )
def compare_activations(names):

    x = torch.linspace(-6, 6, 1200)

    activations = {
        "ReLU": nn.ReLU(),
        "LeakyReLU": nn.LeakyReLU(0.1),
        "ELU": nn.ELU(),
        "GELU": nn.GELU(),
        "SiLU": nn.SiLU(),
        "Mish": nn.Mish(),
        "Sigmoid": nn.Sigmoid(),
        "Tanh": nn.Tanh(),
        "Softplus": nn.Softplus(),
    }

    fig = go.Figure()

    for name in names:

        with torch.no_grad():
            y = activations[name](x)

        fig.add_trace(
            go.Scatter(
                x=x.numpy(),
                y=y.numpy(),
                mode="lines",
                name=name
            )
        )

    fig.update_layout(
        title="Activation Comparison",
        xaxis_title="x",
        yaxis_title="f(x)",
        width=850,
        height=500,
        hovermode="x unified"
    )

    fig.show()

##最重要的激活函数：ReLU


```
class torch.nn.ReLU(inplace=False)
#inplace=True (原地操作)：表示激活函数会直接在输入的Tensor 上进行修改。也就是说，原本的输入数据会被覆盖，不再保留计算前的值，用于节省显存。
#inplace=False (默认值)：表示激活函数会创建一个新的 Tensor 来存储输出结果，原始的输入 Tensor *保持不变。一般设置为False*
```




In [71]:
# @title
activation_card(
    name="ReLU",
    activation=nn.ReLU(),
    formula=r"""ReLU(x)=max(0,x)""",
    description="""
    特点：
    - 正半轴保持线性；
    - 负半轴直接置零；
    - 计算及其简单；
    - 将负值归零，天然容易产生稀疏激活；
    - 可能出现 Dead ReLU（神经元死亡）：
    - 假设某个神经元长期$x<0$,那么$f(x)=f'(x)=0$，这个神经元可能永远更新不过来了。
    """,
    ylim=(-1, 6)
)

##LeakyReLU：解决ReLU负区间梯度为0

```
class torch.nn.LeakyReLU(negative_slope=0.01, inplace=False)
```



In [68]:
activation_card(
    name="LeakyReLU",
    activation=nn.LeakyReLU(negative_slope=0.03, inplace=False),
    formula=r"""PReLU(x)=max(0,x)+negative_{slope}∗min(0,x)=\begin{cases}
    x,&x\geq0\\
    ax,&x<0
    \end{cases}""",
    description="""
    - 和ReLU的区别就在负半轴
    - LeakyReLU的负半轴梯度为很小的正数，所以神经元即使进入负区间，仍然能够得到梯度。
    - 负数没有完全关闭，只是大幅度削弱了,可以避免Dead ReLU.
    """,
    ylim=(-0.5, 6)
)

##PReLU：让负斜率自己学
LeakyReLU的负半轴斜率要自己设置，没有理论答案，因此有了可训练负半轴斜率的PReLU。


```
class torch.nn.PReLU(num_parameters=1, init=0.25, device=None, dtype=None)
#num_parameters:控制学习参数negative_slope的数量，只有两个合法值:1和输入通道数，表示所有通道共用一个negative_slope和每个通道分开训练一个negative_slope。默认为1
#init:negative_slope的初始值，默认为0.25，表示模型刚开始训练时负半轴的默认斜率，在此基础上通过反向传播自动调整。
#device & dtype：指定参数存储的设备以及数据类型，比如cuda;torch.float32。
```



In [67]:
activation_card(
    name="PReLU",
    activation=nn.PReLU(num_parameters=1, init=0.25, device=None, dtype=None),
    formula=r"""PReLU(x)=max(0,x)+negative_{slope}∗min(0,x)=\begin{cases}
    x,&x\geq0\\
    ax,&x<0
    \end{cases}""",
    description="""
    - $a(negative_{slope})$是可训练参数，网络自己决定该保留多少负值：<br>
    - PReLU是少数包含可训练参数的激活函数
    """,
    ylim=(-2, 6)
)

##RReLU：随机版本 LeakyReLU
核心特点：在训练过程中，负半轴的斜率是从一个均匀分布中随机采样得到的。训练模式下每一轮迭代都不同，起到正则化的作用，减少过拟合。测试模式下负半轴斜率固定为上下界的平均值，默认为0.0092


```
class torch.nn.RReLU(lower=0.125, upper=0.3333333333333333, inplace=False)
#lower:均匀分布的下界，默认值0.125
#upper:均匀分布的上界，默认值0.333333

```



In [61]:
activation_card(
    name="RReLU",
    activation=nn.RReLU(lower=0.125, upper=0.3333333333333333, inplace=False),

    formula=r"""f(x)=
    \begin{cases}
    x,&x\geq0\\
    ax,&x<0
    \end{cases}""",

    description="""
    - $ a(negative_{slope})$ 是随机参数
    - 可以看到每个函数点采用每次随机得到的斜率，点分布在上下界之间。
    - 给激活增加随机扰动，形成一定正则化效果,增加网络的泛化能力。
    - 目前比较少用。
    """,
    ylim=(-2.5, 6)
)

##Sigmoid：最经典，但已经很少用于隐藏层

```
class torch.nn.Sigmoid()
#没有参数，不支持原地操作
```
比较适合表达：概率、开关、门控程度。核心问题在于梯度消失，梯度最大点也仅有0.25，其余点或者多次Sigmoid时梯度迅速降到接近0。现代深层网络很少使用 Sigmoid 作为普通 hidden activation，但是在门控或二分类时依旧很重要。




In [72]:
activation_card(
    name="Sigmoid",

    activation=torch.nn.Sigmoid(),

    formula=r"""\sigma(x)=\frac{1}{1+e^{-x}}""",

    description=r"""
### 特点

- 输出范围为 $(0,1)$
- 可以表示概率或者门控程度
- 当 $|x|$ 较大时容易进入饱和区
- 深层网络中容易出现梯度消失
""",

    ylim=(-0.2, 1.2)
)

##Tanh：零中心版 Sigmoid
```
class torch.nn.Tanh()
```
导数最大值为1，比Sigmoid好得多，但是深层网络依然存在梯度消失问题

In [87]:
activation_card(
    name="Tanh",

    activation=torch.nn.Tanh(),

    formula=r"""\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}""",

    description=r"""
### 特点

- 输出范围为 $(-1,1)$
- 常出现于RNN、LSTM、输出限制在$[-1,1]$的模型
- 当 $|x|$ 较大时进入饱和区
- 深层网络中依然容易出现梯度消失
""",

    ylim=(-1.2, 1.2)
)

##GELU：Transformer 时代非常重要

```
class torch.nn.GELU(approximate='none')
#approximate：指定是否使用近似算法来计算GELU，因为GELU的原始定义设计高斯误差函数，计算量较大。默认使用精确公式计算，“tanh”使用基于tanh函数的近似公式，计算更快且误差很小
```
全称Gaussian Error Linear Unit。
相比于ReLU更平滑，在负半轴不完全为0，保留了一点负反馈。

GELU可以理解为$$f(x)=x×"保留概率"$$
$x$很负时$\Phi(x)≈0$,基本删除；$x$很正时$\Phi(x)≈1$,基本保留；在0附近平滑地决定保留多少，小负值不会完全删除，保留了一小部分负值信息。

In [93]:
activation_card(
    name="GELU",

    activation=torch.nn.GELU(),

    formula=r"""GELU(x)=x{\Phi(x)}
\approx
\frac12x
\left[
1+
\tanh
\left(
\sqrt{\frac2\pi}
(x+0.044715x^3)
\right)
\right]""",

    description=r"""
### 特点

- 相当于更平滑、非严格单调、更可以概率解释的ReLU
- 在Transformer领域很常见
- 可以把它视为现代神经网络中最核心的激活函数之一
""",

    ylim=(-1, 6)
)

##SiLU（也叫Swish）：现在同样非常重要


```
class torch.nn.SiLU(inplace=False)
```
和 GELU 的思想非常相似，都相当于$$x×一个平滑门$$


In [96]:
activation_card(
    name="SiLU",

    activation=torch.nn.SiLU(),

    formula=r"""
SiLU(x)=x\sigma(x)=
\frac{x}{1+e^{-x}}
""",

    description=r"""
### 特点

- 思想和GELU类似
- 常见于CNN、YOLO等视觉网络、一些Transformer/LLM架构
- GELU和SiLU是现代模型里非常值得掌握的两个激活函数
""",

    ylim=(-1, 6)
)

##Mish

In [1]:
# @title
import torch
import torch.nn as nn
import plotly.graph_objects as go

# 输入范围
x = torch.linspace(-6, 6, 1200)

activations = {
    "ReLU": nn.ReLU(),
    "LeakyReLU": nn.LeakyReLU(negative_slope=0.1),
    "ELU": nn.ELU(),
    "GELU": nn.GELU(),
    "SiLU / Swish": nn.SiLU(),
    "Mish": nn.Mish(),
    "Sigmoid": nn.Sigmoid(),
    "Tanh": nn.Tanh(),
    "Softplus": nn.Softplus(),
}

fig = go.Figure()

for name, activation in activations.items():
    with torch.no_grad():
        y = activation(x)

    # 默认只打开几个最常用的，其他放在图例中等待点击
    default_visible = name in ["ReLU", "GELU", "SiLU / Swish"]

    fig.add_trace(
        go.Scatter(
            x=x.numpy(),
            y=y.numpy(),
            mode="lines",
            name=name,
            visible=True if default_visible else "legendonly"
        )
    )

fig.update_layout(
    title="常用激活函数比较",
    xaxis_title="x",
    yaxis_title="f(x)",
    width=900,
    height=550,
    hovermode="x unified",
)

fig.update_xaxes(
    range=[-6, 6],
    zeroline=True,
    showgrid=True
)

fig.update_yaxes(
    range=[-3, 6],
    zeroline=True,
    showgrid=True
)

fig.show()